# Fabric Data Agent MCP - User (delegated) auth sample

Straight from the Microsoft Learn doc ["Data agent as Model Context Protocol server"](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-mcp-server#connect-from-python), using `AzureCliCredential` so the token represents **your own signed-in user identity** (delegated auth), not a service principal.

- On your own machine: run `az login` first, then run this notebook - `AzureCliCredential` reuses that sign-in.
- **If you import this into Fabric** and `AzureCliCredential` doesn't pick up a usable session there (no `az login` context inside the Fabric runtime), replace the credential block below with:
  ```python
  from notebookutils import credentials as nb_credentials
  token_value = nb_credentials.getToken("https://api.fabric.microsoft.com")
  def get_auth_headers():
      return {"Authorization": f"Bearer {token_value}"}
  ```
  which uses the notebook's own signed-in user identity when run interactively in the Fabric portal.

In [ ]:
%pip install mcp azure-identity --quiet

In [ ]:
import asyncio

from azure.identity import AzureCliCredential
from mcp import ClientSession

# Defensive import: different mcp package versions/Fabric runtimes expose
# this client under different names.
try:
    from mcp.client.streamable_http import streamablehttp_client
except ImportError:
    from mcp.client.streamable_http import streamable_http_client as streamablehttp_client

workspace_id = "<your-workspace-id>"
data_agent_id = "<your-data-agent-id>"
question = "<your question>"

mcp_url = (
    f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}"
    f"/dataagents/{data_agent_id}/agent"
)

In [ ]:
# AzureCliCredential reuses the sign-in from `az login` - this represents
# your own signed-in USER identity (delegated auth), not a service principal.
credential = AzureCliCredential()


def get_auth_headers():
    token = credential.get_token("https://api.fabric.microsoft.com/.default")
    return {"Authorization": f"Bearer {token.token}"}

In [ ]:
async def query_data_agent(question):
    headers = get_auth_headers()

    async with streamablehttp_client(mcp_url, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # The data agent exposes a single tool. Discover it, then call it.
            tools = await session.list_tools()
            tool = tools.tools[0]
            question_arg = next(iter(tool.inputSchema["properties"]))

            result = await session.call_tool(tool.name, {question_arg: question})

            answers = [block.text for block in result.content if block.type == "text"]
            return "\n".join(answers)

In [ ]:
# Run this cell as-is if you're in a Fabric/Jupyter notebook kernel
# (which already runs its own event loop) - top-level `await` works there.
# If running this as a standalone .py script instead, replace the line
# below with: answer = asyncio.run(query_data_agent(question))
answer = await query_data_agent(question)
print(answer)